In [201]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split,cross_val_score,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report,confusion_matrix, precision_score, recall_score

import pickle

In [202]:
df= pd.read_csv('../data/creditcard.csv')
df.sample(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
154236,100925.0,1.953292,-0.680052,-0.376987,0.115863,-0.281747,0.965405,-1.211798,0.336420,2.636886,...,-0.131274,-0.158162,0.214958,-0.474103,-0.551901,0.462249,-0.058823,-0.056073,39.00,0
42960,41314.0,1.425474,-1.229649,0.660221,-1.550415,-1.479929,0.170570,-1.463155,0.189026,-1.772884,...,-0.004636,0.369406,-0.023068,-0.314914,0.265239,-0.070248,0.060351,0.010430,17.40,0
78220,57425.0,-1.175670,0.737788,1.550711,-0.554641,-0.223433,0.590048,-0.224369,0.948782,-0.144331,...,0.294871,0.902194,-0.116467,-0.246973,-0.187520,0.491324,0.221349,0.090649,17.95,0
74782,55753.0,-2.595275,-2.259936,1.720243,-1.991846,-0.582608,-1.206658,-1.334881,0.578124,-2.310798,...,-0.014050,0.378454,1.121939,0.264812,0.638363,-0.154600,0.005166,-0.176630,40.60,0
216961,140706.0,-0.317115,0.649149,0.571228,0.159155,-0.037876,-1.152326,0.833999,-0.251649,-0.113674,...,-0.194262,-0.672161,0.083486,0.356579,-0.500334,0.307411,-0.272915,0.058242,41.75,0


In [204]:
'''important_features = [
    'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8',
    'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16',
    'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24',
    'V25', 'V26', 'V27', 'V28'
]'''

"important_features = [\n    'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8',\n    'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16',\n    'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24',\n    'V25', 'V26', 'V27', 'V28'\n]"

In [203]:
important_features=['V14', 'V10', 'V12', 'V4', 'V17', 'V3', 'V11', 'V16']

### Results Comparison

| Metric         | 8 Features | 28 Features |
|----------------|------------|-------------|
| Mean Recall    | 0.8140     | 0.8141      |
| Mean F2        | 0.8250     | 0.8298      |
| Mean Precision | 0.8773     | 0.9021      |

### Conclusion
- Recall is virtually identical — the 8 features carry all the predictive signal
- 20 extra features added minor precision gain, no recall improvement


In [205]:
df=df.drop_duplicates()

In [206]:
df=df.drop(columns=['Time'])

In [207]:
rf1=ColumnTransformer([
    ('robust scaler',RobustScaler(),['Amount']),
    ('passthrough_v','passthrough',important_features)
],remainder='drop'
)

In [208]:
X=df.drop(columns=['Class'])
y=df.iloc[:,-1]

In [209]:
X

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,1.475829,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,0.059616,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,0.001396,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,0.127434,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00


In [210]:
xgb=XGBClassifier()

In [211]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,stratify=y)

In [ ]:
X_train=rf1.fit_transform(X_train)
X_test=rf1.transform(X_test)

In [214]:
y_pred=xgb.predict(X_test)

In [215]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9993937855974736

In [216]:
from sklearn.metrics import recall_score,precision_score,f1_score
print(f'recall : {recall_score(y_test,y_pred)}')
print(f'precision : {precision_score(y_test,y_pred)}')
print(f'f1 : {f1_score(y_test,y_pred)}')

recall : 0.7203389830508474
precision : 0.8947368421052632
f1 : 0.7981220657276995


In [217]:
pipe=Pipeline([
    ('scaling',rf1),
    ('xgb',xgb)
])

In [218]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaling', ...), ('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('robust scaler', ...), ('passthrough_v', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tr

In [219]:
scores=cross_val_score(pipe,X,y,cv=5,scoring='recall')
print(f"Recall per fold: {scores}")
print(f"Mean Recall: {scores.mean():.4f}")

Recall per fold: [0.83157895 0.79787234 0.64893617 0.76842105 0.64210526]
Mean Recall: 0.7378


Stratified K-Fold is a cross-validation technique used to evaluate machine learning models, particularly when your dataset has class imbalance.

In [220]:
from sklearn.metrics import make_scorer, fbeta_score

# F2: recall counts 2x more than precision
f2_scorer = make_scorer(fbeta_score, beta=2)


In [221]:
from sklearn.model_selection import StratifiedKFold

In [222]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe, X, y, cv=cv, scoring=f2_scorer)
print(f"F2 per fold: {scores}")
print(f"Mean F2: {scores.mean():.4f}")

F2 per fold: [0.73496659 0.78431373 0.81344902 0.78891258 0.79347826]
Mean F2: 0.7830


In [223]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='recall')
print(f"recall per fold: {scores}")
print(f"Mean recall: {scores.mean():.4f}")

recall per fold: [0.69473684 0.76595745 0.79787234 0.77894737 0.76842105]
Mean recall: 0.7612


> here we used the default xgb without any hyperparametere tuning so we got an avg recall of 0.7378

### now lets try scale_pos_weights

Note that we have neither used smote or scaled the pos weights in xgb to take account of class imbalance

taking ratio of fraud/legit ratio

In [224]:
ratio = (y == 0).sum() / (y == 1).sum()
print(f"Negative / Positive ratio: {ratio:.2f}")

Negative / Positive ratio: 598.84


Nearly 600 negatives for every single positive

In [225]:
xgb_spw = XGBClassifier(
    scale_pos_weight = ratio,
    random_state     = 42,
    eval_metric      = 'aucpr',
)

In [226]:
pipe_spw = Pipeline([
    ('scaling', rf1),
    ('xgb',     xgb_spw)
])

In [227]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe_spw, X, y, cv=cv, scoring='recall')
print(f"recall per fold: {scores}")
print(f"Mean recall: {scores.mean():.4f}")

recall per fold: [0.72631579 0.82978723 0.84042553 0.87368421 0.8       ]
Mean recall: 0.8140


In [228]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe_spw, X, y, cv=cv, scoring=f2_scorer)
print(f"F2 per fold: {scores}")
print(f"Mean F2: {scores.mean():.4f}")

F2 per fold: [0.75824176 0.83690987 0.8512931  0.86638831 0.81196581]
Mean F2: 0.8250


In [229]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe_spw, X, y, cv=cv, scoring='precision')
print(f"precision per fold: {scores}")
print(f"Mean precision: {scores.mean():.4f}")

precision per fold: [0.92       0.86666667 0.89772727 0.83838384 0.86363636]
Mean precision: 0.8773


In [230]:
param_dist = {
    'xgb__n_estimators': [500,600],
    'xgb__max_depth': [6,7,8,9],
    'xgb__learning_rate': [0.1,0.2,0.5],
    'xgb__subsample': [0.6, 0.8, 1.0],
    'xgb__colsample_bytree': [0.6, 0.8, 1.0],
    'xgb__min_child_weight': [1, 5, 10, 50]
}

In [231]:
search = RandomizedSearchCV(
    pipe_spw, param_dist,
    n_iter=30,
    scoring=f2_scorer,   # optimise for F2, not accuracy
    cv=cv,
    random_state=42
)

In [232]:
#search.fit(X, y)
#print(search.best_params_)

{'xgb__subsample': 1.0, 'xgb__n_estimators': 500, 'xgb__min_child_weight': 10, 'xgb__max_depth': 9, 'xgb__learning_rate': 0.1, 'xgb__colsample_bytree': 0.6}

In [233]:
#print(search.best_score_)

0.8371288208108197

In [234]:
xgb_hyperparametertuned = XGBClassifier(
    subsample         = 1.0,
    n_estimators      = 500,
    min_child_weight  = 10,
    max_depth         = 9,
    learning_rate     = 0.1,
    colsample_bytree  = 0.6,
    scale_pos_weight  = ratio,
    random_state      = 42,
    eval_metric       = 'aucpr'
)

In [235]:
pipe_hyperparametertuned=Pipeline([
    ('preprocessor',rf1),
    ('XGBoost Hyperparameter tuned along with scaled position weights',xgb_hyperparametertuned)
])

In [236]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe_hyperparametertuned, X, y, cv=cv, scoring='recall')
print(f"recall per fold: {scores}")
print(f"Mean recall: {scores.mean():.4f}")


recall per fold: [0.74736842 0.82978723 0.84042553 0.89473684 0.82105263]
Mean recall: 0.8267


In [237]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe_hyperparametertuned, X, y, cv=cv, scoring=f2_scorer)
print(f"F2 per fold: {scores}")
print(f"Mean F2: {scores.mean():.4f}")

F2 per fold: [0.77510917 0.84233261 0.8512931  0.88357588 0.83333333]
Mean F2: 0.8371


In [238]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
scores = cross_val_score(pipe_hyperparametertuned, X, y, cv=cv, scoring='precision')
print(f"precision per fold: {scores}")
print(f"Mean precision: {scores.mean():.4f}")

precision per fold: [0.91025641 0.89655172 0.89772727 0.84158416 0.88636364]
Mean precision: 0.8865


In [242]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, random_state=42, stratify=y
)

pipe_hyperparametertuned.fit(X_train_df, y_train)

y_proba = pipe_hyperparametertuned.predict_proba(X_test_df)[:, 1]
y_pred = (y_proba > 0.3).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     70814
           1       0.85      0.77      0.81       118

    accuracy                           1.00     70932
   macro avg       0.93      0.89      0.90     70932
weighted avg       1.00      1.00      1.00     70932



In [243]:
for threshold in [0.1, 0.15, 0.2, 0.25, 0.3]:
    y_pred = (y_proba > threshold).astype(int)
    r = recall_score(y_test, y_pred)
    p = precision_score(y_test, y_pred)
    f2 = fbeta_score(y_test, y_pred, beta=2)
    print(f"Threshold {threshold:.2f} → Recall: {r:.4f} | Precision: {p:.4f} | F2: {f2:.4f}")

Threshold 0.10 → Recall: 0.7966 | Precision: 0.7402 | F2: 0.7846
Threshold 0.15 → Recall: 0.7881 | Precision: 0.7623 | F2: 0.7828
Threshold 0.20 → Recall: 0.7881 | Precision: 0.8378 | F2: 0.7976
Threshold 0.25 → Recall: 0.7712 | Precision: 0.8426 | F2: 0.7845
Threshold 0.30 → Recall: 0.7712 | Precision: 0.8505 | F2: 0.7858


In [245]:
from sklearn.metrics import average_precision_score
print(f"PR-AUC: {average_precision_score(y_test, y_proba):.4f}")



PR-AUC: 0.8032


In [246]:
# Confusion matrix at chosen threshold (0.20)
y_final = (y_proba > 0.20).astype(int)
print(confusion_matrix(y_test, y_final))

[[70796    18]
 [   25    93]]


In [247]:
y_final = (y_proba > 0.10).astype(int)
print(confusion_matrix(y_test, y_final))

[[70781    33]
 [   24    94]]


In [250]:
pipe_hyperparametertuned.threshold = 0.20

with open('../model/fraud_model_using_XGBoost.pkl', 'wb') as f:
    pickle.dump(pipe_hyperparametertuned, f)

print("Model saved as fraud_model.pkl")

Model saved as fraud_model.pkl


# Predict
y_proba = model.predict_proba(X_new)[:, 1]             
y_pred  = (y_proba > model.threshold).astype(int)